In [1]:
!git clone https://github.com/xuebinqin/U-2-Net.git
!cd U-2-Net
!wget https://github.com/xuebinqin/U-2-Net/releases/download/v1.0/u2net.pth

#download: https://drive.google.com/file/d/1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ/view (u2net.pth)

Cloning into 'U-2-Net'...
remote: Enumerating objects: 1077, done.
remote: Counting objects: 100% (422/422), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1077 (delta 398), reused 380 (delta 380), pack-reused 655 (from 2)
Receiving objects: 100% (1077/1077), 66.95 MiB | 21.99 MiB/s, done.
Resolving deltas: 100% (536/536), done.
--2026-07-03 15:48:52--  https://github.com/xuebinqin/U-2-Net/releases/download/v1.0/u2net.pth
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-03 15:48:52 ERROR 404: Not Found.



In [2]:
!git clone https://github.com/xuebinqin/U-2-Net.git

fatal: destination path 'U-2-Net' already exists and is not an empty directory.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import sys
import cv2
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.autograd import Variable

sys.path.append('/content/U-2-Net/model')
from u2net import U2NET

# ---------------------------------------------------------------
# Load U2-Net model
# ---------------------------------------------------------------
model_dir = '/content/drive/MyDrive/Models/u2net.pth'
net = U2NET(3, 1)
net.load_state_dict(torch.load(model_dir, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
net.eval()
if torch.cuda.is_available():
    net.cuda()

input_directory = '/content/drive/MyDrive/logo_input/'
output_directory = '/content/drive/MyDrive/logo_output/'

transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def get_saliency_mask(image, threshold=127):
    """U2-Net saliency mask, thresholded to binary to avoid soft-edge color bleed."""
    transformed_image = transform(image).unsqueeze(0)
    input_var = Variable(transformed_image)
    if torch.cuda.is_available():
        input_var = input_var.cuda()

    with torch.no_grad():
        d1, _, _, _, _, _, _ = net(input_var)

    pred = d1[:, 0, :, :]
    pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)
    pred_np = pred.squeeze().cpu().data.numpy()

    mask = Image.fromarray((pred_np * 255).astype(np.uint8)).resize(image.size, Image.BILINEAR)
    mask_np = np.array(mask)

    # Binary threshold -> crisp edges, no partial-alpha color bleed
    _, sal_mask = cv2.threshold(mask_np, threshold, 255, cv2.THRESH_BINARY)
    return sal_mask


def get_flood_fill_foreground(image, tolerance=30):
    """
    Flood-fills from the image border(s) using color similarity.
    Removes background color wherever it's connected to the border
    (including thin gaps between logo elements), but leaves anything
    enclosed / disconnected from the border untouched (e.g. text sitting
    inside or on top of the logo).
    """
    img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    h, w = img_cv.shape[:2]
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)

    filled = img_cv.copy()
    seed_points = [
        (0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1),
        (w // 2, 0), (0, h // 2), (w - 1, h // 2), (w // 2, h - 1),
    ]
    for seed in seed_points:
        if flood_mask[seed[1] + 1, seed[0] + 1] == 0:
            cv2.floodFill(
                filled, flood_mask, seed, (255, 255, 255),
                loDiff=(tolerance, tolerance, tolerance),
                upDiff=(tolerance, tolerance, tolerance),
                flags=4 | cv2.FLOODFILL_FIXED_RANGE,
            )

    bg_mask = (flood_mask[1:-1, 1:-1] * 255).astype(np.uint8)
    fg_mask = cv2.bitwise_not(bg_mask)
    return fg_mask


def remove_background(image_path, output_path, color_tolerance=30, sal_threshold=127):
    image = Image.open(image_path).convert('RGB')

    sal_mask = get_saliency_mask(image, threshold=sal_threshold)
    flood_fg_mask = get_flood_fill_foreground(image, tolerance=color_tolerance)

    # Keep a pixel if EITHER detector calls it foreground.
    # This protects text/fine detail that saliency might miss,
    # while flood-fill still strips connected background color.
    combined = cv2.bitwise_or(sal_mask, flood_fg_mask)

    # Clean up: close small holes, remove speckle noise
    kernel = np.ones((3, 3), np.uint8)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=2)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel, iterations=1)

    # Slight feather so edges aren't jagged
    combined = cv2.GaussianBlur(combined, (3, 3), 0)

    rgba = np.array(image.convert('RGBA'))
    rgba[:, :, 3] = combined
    Image.fromarray(rgba, 'RGBA').save(output_path, format='PNG')


# ---------------------------------------------------------------
# Batch process
# ---------------------------------------------------------------
i = 0
for root, dirs, files in os.walk(input_directory):
    for filename in files:
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.jfif', '.gif')):
            input_path = os.path.join(root, filename)
            relative_path = os.path.relpath(root, input_directory)
            output_folder = os.path.join(output_directory, relative_path)
            os.makedirs(output_folder, exist_ok=True)
            output_filename = os.path.splitext(filename)[0] + '.png'
            output_path = os.path.join(output_folder, output_filename)
            remove_background(input_path, output_path)
            i += 1
            print(f'Image processing: {i} - Saved: {output_path}')

print("Processing complete")

/content/U-2-Net/model/u2net.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/tmp/ipykernel_3805/2818456380.py:107: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, 'RGBA').save(output_path, format='PNG')


Image processing: 1 - Saved: /content/drive/MyDrive/logo_output/./images (2).png
Image processing: 2 - Saved: /content/drive/MyDrive/logo_output/./images (3).png
Image processing: 3 - Saved: /content/drive/MyDrive/logo_output/./images (4).png
Image processing: 4 - Saved: /content/drive/MyDrive/logo_output/./images (5).png
Processing complete
